# Data Preprocessing


Import libraries and paths

In [2]:
import os
import numpy as np
import pandas as pd

from IPython.display import display

# Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Project paths
BASE_DIR = "/content/drive/MyDrive/ML Assignment"
SPLIT_DIR = os.path.join(BASE_DIR, "splitted_dataset")

print("Project directory:", BASE_DIR)
print("Split directory:", SPLIT_DIR)
print("Files:", os.listdir(SPLIT_DIR))

Mounted at /content/drive
Project directory: /content/drive/MyDrive/ML Assignment
Split directory: /content/drive/MyDrive/ML Assignment/splitted_dataset
Files: ['train.csv', 'validation.csv', 'test.csv']


Load Splitted datasets

In [3]:
train_df = pd.read_csv(
    os.path.join(SPLIT_DIR, "train.csv")
)

val_df = pd.read_csv(
    os.path.join(SPLIT_DIR, "validation.csv")
)

test_df = pd.read_csv(
    os.path.join(SPLIT_DIR, "test.csv")
)

print("Training shape   :", train_df.shape)
print("Validation shape :", val_df.shape)
print("Test shape       :", test_df.shape)

Training shape   : (18183, 121)
Validation shape : (3873, 121)
Test shape       : (3923, 121)


In [4]:
# Preserve match IDs separately for safe alignment in feature engineering

train_ids = train_df["id"].copy()
val_ids = val_df["id"].copy()
test_ids = test_df["id"].copy()

# Verify IDs are valid
assert train_ids.is_unique, "Duplicate match IDs found in training data."
assert val_ids.is_unique, "Duplicate match IDs found in validation data."
assert test_ids.is_unique, "Duplicate match IDs found in test data."

assert len(train_ids) == len(train_df)
assert len(val_ids) == len(val_df)
assert len(test_ids) == len(test_df)

print("Match IDs preserved successfully.")
print("Training IDs   :", len(train_ids))
print("Validation IDs :", len(val_ids))
print("Test IDs       :", len(test_ids))

Match IDs preserved successfully.
Training IDs   : 18183
Validation IDs : 3873
Test IDs       : 3923


Verify the target

In [5]:
TARGET = "target_outcome"

print("Target column:", TARGET)

for name, df in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name} target distribution:")
    display(
        df[TARGET]
        .value_counts()
        .rename_axis("Outcome")
        .reset_index(name="Count")
    )

Target column: target_outcome

Training target distribution:


,Outcome,Count
0,Home Win,8433
1,Away Win,5126
2,Draw,4624



Validation target distribution:


,Outcome,Count
0,Home Win,1753
1,Away Win,1147
2,Draw,973



Test target distribution:


,Outcome,Count
0,Home Win,1731
1,Away Win,1193
2,Draw,999


Verify chronological split

In [6]:
for name, df in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    print(f"{name}:")
    print("  Start:", df["date"].min())
    print("  End  :", df["date"].max())

Training:
  Start: 2008-07-18 00:00:00
  End  : 2014-02-10 00:00:00
Validation:
  Start: 2014-02-11 00:00:00
  End  : 2015-04-10 00:00:00
Test:
  Start: 2015-04-11 00:00:00
  End  : 2016-05-25 00:00:00


In [7]:
assert train_df["date"].max() < val_df["date"].min(),  "Train/validation dates overlap"
assert val_df["date"].max()   < test_df["date"].min(), "Validation/test dates overlap"

print("Training end < Validation start: True")
print("Validation end < Test start: True")

Training end < Validation start: True
Validation end < Test start: True


In [8]:
for boundary_date in [
    val_df["date"].min(),
    test_df["date"].min()
]:
    print("=" * 60)
    print("Boundary date:", boundary_date)

    print(
        "Training:",
        (train_df["date"] == boundary_date).sum()
    )

    print(
        "Validation:",
        (val_df["date"] == boundary_date).sum()
    )

    print(
        "Test:",
        (test_df["date"] == boundary_date).sum()
    )

Boundary date: 2014-02-11 00:00:00
Training: 0
Validation: 6
Test: 0
Boundary date: 2015-04-11 00:00:00
Training: 0
Validation: 0
Test: 38


Create explicit leakage list

In [9]:
EVENT_COLUMNS = ["goal", "shoton", "shotoff", "foulcommit",
                 "card", "cross", "corner", "possession"]

LEAKAGE_COLUMNS = [
    "home_team_goal",
    "away_team_goal",
    "target_outcome",
    "total_goals",
    "goal_difference"
] + EVENT_COLUMNS

print("Leakage columns:")
for col in LEAKAGE_COLUMNS:
    print("-", col)

Leakage columns:
- home_team_goal
- away_team_goal
- target_outcome
- total_goals
- goal_difference
- goal
- shoton
- shotoff
- foulcommit
- card
- cross
- corner
- possession


Identify non-predictive columns

In [10]:
id_columns = [
    col for col in train_df.columns
    if col.lower() == "id"
    or col.lower().endswith("_id")
    or col.lower().endswith("_api_id")
    or (col.lower().startswith(("home_player_", "away_player_"))
        and col.split("_")[-1].isdigit())      # player API IDs (home_player_1 ... away_player_11)
]

print("Potential identifier columns:")
for col in id_columns:
    print("-", col)

Potential identifier columns:
- id
- country_id
- league_id
- match_api_id
- home_team_api_id
- away_team_api_id
- home_player_1
- home_player_2
- home_player_3
- home_player_4
- home_player_5
- home_player_6
- home_player_7
- home_player_8
- home_player_9
- home_player_10
- home_player_11
- away_player_1
- away_player_2
- away_player_3
- away_player_4
- away_player_5
- away_player_6
- away_player_7
- away_player_8
- away_player_9
- away_player_10
- away_player_11
- match_id


In [11]:
id_summary = []

for col in id_columns:
    id_summary.append({
        "Column": col,
        "Unique Values": train_df[col].nunique(),
        "Training Rows": len(train_df),
        "Unique Ratio": round(train_df[col].nunique() / len(train_df), 4)
    })

display(pd.DataFrame(id_summary))

,Column,Unique Values,Training Rows,Unique Ratio
0,id,18183,18183,1.0000
1,country_id,11,18183,0.0006
2,league_id,11,18183,0.0006
3,match_api_id,18183,18183,1.0000
4,home_team_api_id,274,18183,0.0151
5,away_team_api_id,274,18183,0.0151
6,home_player_1,728,18183,0.0400
7,home_player_2,1896,18183,0.1043
8,home_player_3,1877,18183,0.1032
9,home_player_4,2032,18183,0.1118


In [12]:
PLAYER_ID_COLUMNS = [f"{s}_player_{i}" for s in ("home", "away") for i in range(1, 12)]

IDENTIFIER_COLUMNS = [
    "id",
    "match_id",
    "country_id",
    "league_id",
    "match_api_id",
    "home_team_api_id",
    "away_team_api_id"
] + PLAYER_ID_COLUMNS

IDENTIFIER_COLUMNS = [
    col for col in IDENTIFIER_COLUMNS
    if col in train_df.columns
]

print("Identifiers selected for removal:")
for col in IDENTIFIER_COLUMNS:
    print("-", col)

Identifiers selected for removal:
- id
- match_id
- country_id
- league_id
- match_api_id
- home_team_api_id
- away_team_api_id
- home_player_1
- home_player_2
- home_player_3
- home_player_4
- home_player_5
- home_player_6
- home_player_7
- home_player_8
- home_player_9
- home_player_10
- home_player_11
- away_player_1
- away_player_2
- away_player_3
- away_player_4
- away_player_5
- away_player_6
- away_player_7
- away_player_8
- away_player_9
- away_player_10
- away_player_11


Create preprocessing exclusion list

In [13]:
EXCLUDE_COLUMNS = sorted(
    set(LEAKAGE_COLUMNS + IDENTIFIER_COLUMNS)
    .intersection(train_df.columns)
)

print("Total columns excluded:", len(EXCLUDE_COLUMNS))

for col in EXCLUDE_COLUMNS:
    print("-", col)

Total columns excluded: 40
- away_player_1
- away_player_10
- away_player_11
- away_player_2
- away_player_3
- away_player_4
- away_player_5
- away_player_6
- away_player_7
- away_player_8
- away_player_9
- away_team_api_id
- away_team_goal
- card
- corner
- country_id
- cross
- foulcommit
- goal
- home_player_1
- home_player_10
- home_player_11
- home_player_2
- home_player_3
- home_player_4
- home_player_5
- home_player_6
- home_player_7
- home_player_8
- home_player_9
- home_team_api_id
- home_team_goal
- id
- league_id
- match_api_id
- match_id
- possession
- shotoff
- shoton
- target_outcome


Create working datasets

In [14]:
X_train_raw = train_df.drop(columns=EXCLUDE_COLUMNS).copy()
X_val_raw = val_df.drop(columns=EXCLUDE_COLUMNS).copy()
X_test_raw = test_df.drop(columns=EXCLUDE_COLUMNS).copy()

print("Training:", X_train_raw.shape)
print("Validation:", X_val_raw.shape)
print("Test:", X_test_raw.shape)

Training: (18183, 81)
Validation: (3873, 81)
Test: (3923, 81)


In [15]:
# Separately store y
y_train = train_df[TARGET].copy()
y_val = val_df[TARGET].copy()
y_test = test_df[TARGET].copy()

print("X_train:", X_train_raw.shape)
print("y_train:", y_train.shape)

X_train: (18183, 81)
y_train: (18183,)


Check remaining columns

In [16]:
column_summary = pd.DataFrame({
    "Column": X_train_raw.columns,
    "Data Type": X_train_raw.dtypes.astype(str),
    "Missing Values": X_train_raw.isnull().sum().values,
    "Missing %": (
        X_train_raw.isnull().mean() * 100
    ).round(2).values,
    "Unique Values": X_train_raw.nunique().values
})

display(column_summary)

,Column,Data Type,Missing Values,Missing %,Unique Values
season,season,object,0,0.00,6
stage,stage,int64,0,0.00,38
date,date,datetime64[ns],0,0.00,1198
home_player_X1,home_player_X1,float64,1821,10.01,3
home_player_X2,home_player_X2,float64,1821,10.01,8
...,...,...,...,...,...
BSA,BSA,float64,4022,22.12,96
league_name,league_name,object,0,0.00,11
country_name,country_name,object,0,0.00,11
home_team,home_team,object,0,0.00,272


Missing-value assessment

In [17]:
def missing_report(df, name):
    result = pd.DataFrame({
        "Column": df.columns,
        "Missing Values": df.isnull().sum().values,
        "Missing %": (df.isnull().mean() * 100).round(2).values
    })

    result = result[result["Missing Values"] > 0]
    result = result.sort_values("Missing %", ascending=False)

    print(f"\n{name} missing-value report")
    display(result)

missing_report(X_train_raw, "Training")
missing_report(X_val_raw, "Validation")
missing_report(X_test_raw, "Test")


Training missing-value report


,Column,Missing Values,Missing %
61,PSA,13779,75.78
59,PSH,13779,75.78
60,PSD,13779,75.78
74,BSH,4022,22.12
75,BSD,4022,22.12
...,...,...,...
17,away_player_X4,1832,10.08
3,home_player_X1,1821,10.01
4,home_player_X2,1821,10.01
26,home_player_Y2,1821,10.01



Validation missing-value report


,Column,Missing Values,Missing %
72,GBD,3873,100.00
71,GBH,3873,100.00
75,BSD,3873,100.00
76,BSA,3873,100.00
74,BSH,3873,100.00
73,GBA,3873,100.00
67,SJA,2562,66.15
66,SJD,2562,66.15
65,SJH,2562,66.15
54,IWD,525,13.56



Test missing-value report


,Column,Missing Values,Missing %
74,BSH,3923,100.00
73,GBA,3923,100.00
75,BSD,3923,100.00
65,SJH,3923,100.00
67,SJA,3923,100.00
76,BSA,3923,100.00
72,GBD,3923,100.00
71,GBH,3923,100.00
66,SJD,3923,100.00
54,IWD,513,13.08


Compare missing-value patterns

In [18]:
missing_comparison = pd.DataFrame({
    "Training Missing": X_train_raw.isnull().sum(),
    "Validation Missing": X_val_raw.isnull().sum(),
    "Test Missing": X_test_raw.isnull().sum()
})

missing_comparison["Total Missing"] = (
    missing_comparison["Training Missing"]
    + missing_comparison["Validation Missing"]
    + missing_comparison["Test Missing"]
)

display(
    missing_comparison.sort_values(
        "Total Missing",
        ascending=False
    )
)

,Training Missing,Validation Missing,Test Missing,Total Missing
PSA,13779,521,511,14811
PSD,13779,521,511,14811
PSH,13779,521,511,14811
BSD,4022,3873,3923,11818
BSH,4022,3873,3923,11818
...,...,...,...,...
date,0,0,0,0
league_name,0,0,0,0
country_name,0,0,0,0
home_team,0,0,0,0


Check categorical variables

In [19]:
categorical_columns = X_train_raw.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_columns = X_train_raw.select_dtypes(
    include=np.number
).columns.tolist()

print("Categorical columns:", len(categorical_columns))
print(categorical_columns)

print("\nNumerical columns:", len(numerical_columns))
print(numerical_columns)

Categorical columns: 5
['season', 'league_name', 'country_name', 'home_team', 'away_team']

Numerical columns: 75
['stage', 'home_player_X1', 'home_player_X2', 'home_player_X3', 'home_player_X4', 'home_player_X5', 'home_player_X6', 'home_player_X7', 'home_player_X8', 'home_player_X9', 'home_player_X10', 'home_player_X11', 'away_player_X1', 'away_player_X2', 'away_player_X3', 'away_player_X4', 'away_player_X5', 'away_player_X6', 'away_player_X7', 'away_player_X8', 'away_player_X9', 'away_player_X10', 'away_player_X11', 'home_player_Y1', 'home_player_Y2', 'home_player_Y3', 'home_player_Y4', 'home_player_Y5', 'home_player_Y6', 'home_player_Y7', 'home_player_Y8', 'home_player_Y9', 'home_player_Y10', 'home_player_Y11', 'away_player_Y1', 'away_player_Y2', 'away_player_Y3', 'away_player_Y4', 'away_player_Y5', 'away_player_Y6', 'away_player_Y7', 'away_player_Y8', 'away_player_Y9', 'away_player_Y10', 'away_player_Y11', 'B365H', 'B365D', 'B365A', 'BWH', 'BWD', 'BWA', 'IWH', 'IWD', 'IWA', 'LBH', 

Check categorical consistency

In [20]:
for col in categorical_columns:
    print("=" * 80)
    print("COLUMN:", col)
    print("Training unique:", X_train_raw[col].nunique())
    print("Validation unique:", X_val_raw[col].nunique())
    print("Test unique:", X_test_raw[col].nunique())

    print("\nSample training values:")
    print(X_train_raw[col].dropna().unique()[:20])

COLUMN: season
Training unique: 6
Validation unique: 2
Test unique: 2

Sample training values:
['2008/2009' '2009/2010' '2010/2011' '2011/2012' '2012/2013' '2013/2014']
COLUMN: league_name
Training unique: 11
Validation unique: 11
Test unique: 11

Sample training values:
['Switzerland Super League' 'Poland Ekstraklasa' 'France Ligue 1'
 'Scotland Premier League' 'Germany 1. Bundesliga'
 'England Premier League' 'Belgium Jupiler League'
 'Portugal Liga ZON Sagres' 'Netherlands Eredivisie' 'Italy Serie A'
 'Spain LIGA BBVA']
COLUMN: country_name
Training unique: 11
Validation unique: 11
Test unique: 11

Sample training values:
['Switzerland' 'Poland' 'France' 'Scotland' 'Germany' 'England' 'Belgium'
 'Portugal' 'Netherlands' 'Italy' 'Spain']
COLUMN: home_team
Training unique: 272
Validation unique: 211
Test unique: 211

Sample training values:
['BSC Young Boys' 'FC Aarau' 'FC Luzern' 'Neuchâtel Xamax' 'AC Bellinzona'
 'FC Basel' 'FC Zürich' 'FC Sion' 'FC Vaduz' 'Grasshopper Club Zürich'


Check numeric columns for impossible values

In [21]:
numeric_summary = X_train_raw[numerical_columns].describe().T
numeric_summary["Missing"] = X_train_raw[numerical_columns].isnull().sum()

display(numeric_summary)

,count,mean,std,min,25%,50%,75%,max,Missing
stage,18183.0,17.523181,10.253143,1.00,9.00,17.0,26.00,38.0,0
home_player_X1,16362.0,0.999389,0.027075,0.00,1.00,1.0,1.00,2.0,1821
home_player_X2,16362.0,2.052255,0.304834,0.00,2.00,2.0,2.00,8.0,1821
home_player_X3,16351.0,4.037368,0.306122,2.00,4.00,4.0,4.00,8.0,1832
home_player_X4,16351.0,6.059385,0.364422,2.00,6.00,6.0,6.00,8.0,1832
...,...,...,...,...,...,...,...,...,...
GBD,14162.0,3.648189,0.867440,1.45,3.20,3.3,3.75,11.0,4021
GBA,14162.0,4.353097,3.010189,1.12,2.50,3.4,5.00,34.0,4021
BSH,14161.0,2.497894,1.507793,1.04,1.67,2.1,2.62,17.0,4022
BSD,14161.0,3.660742,0.868272,1.33,3.25,3.4,3.75,13.0,4022


Date consistency

In [22]:
# Dates were already parsed to datetime in the chronological-split check above
print("Missing dates:")
print("Train:", X_train_raw["date"].isna().sum())
print("Validation:", X_val_raw["date"].isna().sum())
print("Test:", X_test_raw["date"].isna().sum())

Missing dates:
Train: 0
Validation: 0
Test: 0


Missing-value handling

In [23]:
# Copy datasets
X_train = X_train_raw.copy()
X_val = X_val_raw.copy()
X_test = X_test_raw.copy()

# Define all 10 bookmaker odds columns to exclude from raw median imputation
BOOKMAKER_TRIPLETS = {
    "B365": ("B365H", "B365D", "B365A"),
    "BW":   ("BWH",   "BWD",   "BWA"),
    "IW":   ("IWH",   "IWD",   "IWA"),
    "LB":   ("LBH",   "LBD",   "LBA"),
    "PS":   ("PSH",   "PSD",   "PSA"),
    "WH":   ("WHH",   "WHD",   "WHA"),
    "SJ":   ("SJH",   "SJD",   "SJA"),
    "VC":   ("VCH",   "VCD",   "VCA"),
    "GB":   ("GBH",   "GBD",   "GBA"),
    "BS":   ("BSH",   "BSD",   "BSA"),
}
ODDS_COLS = [c for t in BOOKMAKER_TRIPLETS.values() for c in t]

# Numerical columns (EXCLUDING raw odds)
numeric_columns = [
    c for c in X_train.select_dtypes(include=np.number).columns.tolist()
    if c not in ODDS_COLS
]

# Categorical columns
categorical_columns = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical columns (excluding odds):", len(numeric_columns))
print("Categorical columns:", len(categorical_columns))

Numerical columns (excluding odds): 45
Categorical columns: 5


In [24]:
# Numerical imputation values learned from training data (odds remain NaN)
numeric_fill_values = X_train[numeric_columns].median()

# Categorical imputation values learned from training data
categorical_fill_values = {
    col: X_train[col].mode(dropna=True)[0]
    for col in categorical_columns
    if X_train[col].notna().any()
}

assert not any(c in X_train.columns for c in EVENT_COLUMNS + PLAYER_ID_COLUMNS), \
    "Event/player-ID columns must be removed before imputation"

In [25]:
# Apply numerical imputation (only to non-odds numeric columns)
for col in numeric_columns:
    X_train[col] = X_train[col].fillna(numeric_fill_values[col])
    X_val[col] = X_val[col].fillna(numeric_fill_values[col])
    X_test[col] = X_test[col].fillna(numeric_fill_values[col])

# Apply categorical imputation
for col, value in categorical_fill_values.items():
    X_train[col] = X_train[col].fillna(value)
    X_val[col] = X_val[col].fillna(value)
    X_test[col] = X_test[col].fillna(value)

Verify missing values after preprocessing

In [26]:
# 1) Non-odds columns must be completely imputed
for name, d in [("Train", X_train), ("Validation", X_val), ("Test", X_test)]:
    left = d.drop(columns=ODDS_COLS).isnull().sum().sum()
    print(f"{name}: non-odds missing = {left}")
    assert left == 0

# 2) Odds availability per bookmaker and split (home-odds column used as proxy)
book_missing = pd.DataFrame({
    book: {"train %": X_train[h].isna().mean() * 100,
           "val %":   X_val[h].isna().mean() * 100,
           "test %":  X_test[h].isna().mean() * 100}
    for book, (h, _, _) in BOOKMAKER_TRIPLETS.items()
}).T.round(1)
display(book_missing)

UNSTABLE_BOOKS = book_missing[(book_missing > 50).any(axis=1)].index.tolist()
print("Bookmakers unusable/unstable across splits:", UNSTABLE_BOOKS)   # expected: SJ, GB, BS, PS

Train: non-odds missing = 0
Validation: non-odds missing = 0
Test: non-odds missing = 0


,train %,val %,test %
B365,13.1,13.1,12.8
BW,13.2,13.1,12.8
IW,13.3,13.6,13.1
LB,13.2,13.3,12.8
PS,75.8,13.5,13.0
WH,13.2,13.1,12.8
SJ,13.2,66.2,100.0
VC,13.2,13.1,12.8
GB,22.1,100.0,100.0
BS,22.1,100.0,100.0


Bookmakers unusable/unstable across splits: ['PS', 'SJ', 'GB', 'BS']


In [27]:
print("Training target:")
display(y_train.value_counts())

print("\nValidation target:")
display(y_val.value_counts())

print("\nTest target:")
display(y_test.value_counts())

Training target:


,count
target_outcome,
Home Win,8433
Away Win,5126
Draw,4624



Validation target:


,count
target_outcome,
Home Win,1753
Away Win,1147
Draw,973



Test target:


,count
target_outcome,
Home Win,1731
Away Win,1193
Draw,999


Verify the target wasn't accidentally changed

In [28]:
print("Training target:")
display(y_train.value_counts())

print("\nValidation target:")
display(y_val.value_counts())

print("\nTest target:")
display(y_test.value_counts())

Training target:


,count
target_outcome,
Home Win,8433
Away Win,5126
Draw,4624



Validation target:


,count
target_outcome,
Home Win,1753
Away Win,1147
Draw,973



Test target:


,count
target_outcome,
Home Win,1731
Away Win,1193
Draw,999


Final preprocessing summary

In [29]:
preprocessing_summary = pd.DataFrame({
    "Dataset": [
        "Training",
        "Validation",
        "Test"
    ],
    "Rows": [
        len(X_train),
        len(X_val),
        len(X_test)
    ],
    "Columns": [
        X_train.shape[1],
        X_val.shape[1],
        X_test.shape[1]
    ],
    "Missing Values (odds only, kept as NaN)": [
        X_train[ODDS_COLS].isnull().sum().sum(),
        X_val[ODDS_COLS].isnull().sum().sum(),
        X_test[ODDS_COLS].isnull().sum().sum()
    ],
    "Missing Values (non-odds)": [
        X_train.drop(columns=ODDS_COLS).isnull().sum().sum(),
        X_val.drop(columns=ODDS_COLS).isnull().sum().sum(),
        X_test.drop(columns=ODDS_COLS).isnull().sum().sum()
    ]
})

display(preprocessing_summary)

,Dataset,Rows,Columns,"Missing Values (odds only, kept as NaN)",Missing Values (non-odds)
0,Training,18183,81,115851,0
1,Validation,3873,81,41700,0
2,Test,3923,81,45909,0


In [30]:
# Final ID alignment verification

assert len(train_ids) == len(X_train) == len(y_train)
assert len(val_ids) == len(X_val) == len(y_val)
assert len(test_ids) == len(X_test) == len(y_test)

# Verify the preprocessing stage did not change row order
assert train_ids.reset_index(drop=True).equals(
    train_df["id"].reset_index(drop=True)
)

assert val_ids.reset_index(drop=True).equals(
    val_df["id"].reset_index(drop=True)
)

assert test_ids.reset_index(drop=True).equals(
    test_df["id"].reset_index(drop=True)
)

print("ID and row alignment verified before saving.")

ID and row alignment verified before saving.


In [31]:
# Verify raw odds kept their genuine missing rates in EVERY split (not only training)
for c in ["SJH", "GBH", "BSH", "PSH"]:
    print(
        f"{c} NaN rate -> train: {X_train[c].isna().mean():.3f} | "
        f"val: {X_val[c].isna().mean():.3f} | test: {X_test[c].isna().mean():.3f}"
    )

# Imputation must not have touched any odds column
for name, imputed, raw in [("Train", X_train, X_train_raw),
                           ("Validation", X_val, X_val_raw),
                           ("Test", X_test, X_test_raw)]:
    assert imputed[ODDS_COLS].isna().equals(raw[ODDS_COLS].isna()), f"{name}: odds were altered"

print("\nUnstable bookmakers (>50% missing in at least one split):", UNSTABLE_BOOKS)

SJH NaN rate -> train: 0.132 | val: 0.662 | test: 1.000
GBH NaN rate -> train: 0.221 | val: 1.000 | test: 1.000
BSH NaN rate -> train: 0.221 | val: 1.000 | test: 1.000
PSH NaN rate -> train: 0.758 | val: 0.135 | test: 0.130

Unstable bookmakers (>50% missing in at least one split): ['PS', 'SJ', 'GB', 'BS']


Save preprocessed dataset

In [32]:
PREPROCESSED_DIR = os.path.join(
    BASE_DIR,
    "preprocessed_dataset"
)

os.makedirs(PREPROCESSED_DIR, exist_ok=True)

X_train.to_csv(
    os.path.join(PREPROCESSED_DIR, "X_train_preprocessed.csv"), index=False
)

X_val.to_csv(
    os.path.join(PREPROCESSED_DIR, "X_validation_preprocessed.csv"), index=False
)

X_test.to_csv(
    os.path.join(PREPROCESSED_DIR, "X_test_preprocessed.csv"), index=False
)

y_train.to_csv(
    os.path.join(PREPROCESSED_DIR, "y_train.csv"), index=False
)

y_val.to_csv(
    os.path.join(PREPROCESSED_DIR, "y_validation.csv"), index=False
)

y_test.to_csv(
    os.path.join(PREPROCESSED_DIR, "y_test.csv"), index=False
)

# Save match IDs separately

train_ids.rename("id").to_csv(
    os.path.join(
        PREPROCESSED_DIR,
        "train_ids.csv"
    ),
    index=False
)

val_ids.rename("id").to_csv(
    os.path.join(
        PREPROCESSED_DIR,
        "validation_ids.csv"
    ),
    index=False
)

test_ids.rename("id").to_csv(
    os.path.join(
        PREPROCESSED_DIR,
        "test_ids.csv"
    ),
    index=False
)

print("Preprocessed datasets saved to:")
print(PREPROCESSED_DIR)

print("\nFiles:")
print(os.listdir(PREPROCESSED_DIR))

Preprocessed datasets saved to:
/content/drive/MyDrive/ML Assignment/preprocessed_dataset

Files:
['X_train_preprocessed.csv', 'X_validation_preprocessed.csv', 'X_test_preprocessed.csv', 'y_train.csv', 'y_validation.csv', 'y_test.csv', 'train_ids.csv', 'validation_ids.csv', 'test_ids.csv']
